In [ ]:
from typing import Callable
import torch
import sys
sys.path.append('../')
from utils import DataHandler, Metric, find_best_result_from_results_list, set_seed
from utils.grid_search import MatrixNormalization
import matplotlib.pyplot as plt
import tqdm

interaction_data = 'games'
semantic_data = 'qwen'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
normalization_method = 'zscore'
primary_metric = 'NDCG@20'
plot_metric = 'Recall@20'
seed = 42
gamma_list = [i * 0.01 for i in range(101)]

set_seed(seed)

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from typing import Literal
import time
import torch.nn as nn
import torch.nn.functional as F

from models import SGFCF, TrainableModelBase
from utils.trainer import train_epoch, TrainingDebugInfo

BEST_SGFCF_PARAMS = {
    'books': {'k': 2500, 'beta_1': 1.0, 'beta_2': 1.0, 'alpha': 0.0, 'eps': 0.26, 'gamma': 2.5},
    'games': {'k': 130, 'beta_1': 1.0, 'beta_2': 1.2, 'alpha': 19.0, 'eps': 0.5, 'gamma': 0.5},
    'toys': {'k': 440, 'beta_1': 0.5, 'beta_2': 1.7, 'alpha': 9.0, 'eps': 0.5, 'gamma': 2.5},
}


def build_sgfcf_model(datahandler, interaction_data='games', device='cuda', **overrides):
    params = dict(BEST_SGFCF_PARAMS[interaction_data])
    for key, value in overrides.items():
        if value is not None:
            params[key] = value

    cf_model = SGFCF(
        rate_matrix=datahandler.rate_matrix,
        model_config=SGFCF.ModelConfig(
            k=params['k'],
            beta_1=params['beta_1'],
            beta_2=params['beta_2'],
            alpha=params['alpha'],
            eps=params['eps'],
            gamma=params['gamma'],
            device=device,
        ),
    )
    return cf_model, params


datahandler = DataHandler(
    interaction_data=interaction_data,
    semantic_data=semantic_data,
    device=device,
    seed=seed,
)
metric = Metric(device=device)

cf_model, cf_model_params = build_sgfcf_model(
    datahandler=datahandler,
    interaction_data=interaction_data,
    device=device,
)

# SGFCF 没有 predict_rate_matrix() 方法，这里统一用这个变量名保存 CF 预测矩阵
predict_rate_matrix = cf_model.predict()
cf_pred_matrix = predict_rate_matrix

semantic_embeddings = torch.cat(
    [datahandler.user_semantic_embeddings, datahandler.item_semantic_embeddings],
    dim=0,
)

print(f'interaction_data={interaction_data}, semantic_data={semantic_data}, device={device}')
print(f'SGFCF params: {cf_model_params}')
print(f'CF pred matrix shape: {tuple(cf_pred_matrix.shape)}')

In [ ]:
def build_semantic_matrix_from_filtered_s(
    U_user: torch.Tensor,
    U_item: torch.Tensor,
    filtered_S: torch.Tensor,
) -> torch.Tensor:
    return (U_user * filtered_S) @ (U_item * filtered_S).T


def get_normalized_cf_pred_matrix(
    cf_pred_matrix: torch.Tensor,
    normalization_method: str = 'zscore',
) -> torch.Tensor:
    normalize_fn = MatrixNormalization.get_method(normalization_method)
    return normalize_fn(cf_pred_matrix)


def evaluate_cf_semantic_with_gamma(
    cf_pred_matrix: torch.Tensor,
    semantic_pred_matrix: torch.Tensor,
    gamma: float,
    train_rate_matrix: torch.Tensor,
    eval_rate_matrix: torch.Tensor,
    metric: Metric,
    normalization_method: str = 'zscore',
    normalized_cf_pred_matrix: torch.Tensor | None = None,
) -> dict[str, float]:
    normalize_fn = MatrixNormalization.get_method(normalization_method)
    if normalized_cf_pred_matrix is None:
        normalized_cf_pred_matrix = normalize_fn(cf_pred_matrix)
    normalized_semantic_pred_matrix = normalize_fn(semantic_pred_matrix)
    pred_matrix = (1 - gamma) * normalized_cf_pred_matrix + gamma * normalized_semantic_pred_matrix
    return metric.eval(
        train_matrix=train_rate_matrix,
        pred_matrix=pred_matrix,
        test_matrix=eval_rate_matrix,
    )


def search_best_gamma(
    cf_pred_matrix: torch.Tensor,
    semantic_pred_matrix: torch.Tensor,
    train_rate_matrix: torch.Tensor,
    eval_rate_matrix: torch.Tensor,
    metric: Metric,
    normalization_method: str = 'zscore',
    primary_metric: str = 'NDCG@20',
    gamma_list: list[float] | None = None,
    normalized_cf_pred_matrix: torch.Tensor | None = None,
) -> tuple[float, dict[str, float], list[dict[str, float]]]:
    if gamma_list is None:
        gamma_list = [i * 0.01 for i in range(101)]

    normalize_fn = MatrixNormalization.get_method(normalization_method)
    if normalized_cf_pred_matrix is None:
        normalized_cf_pred_matrix = normalize_fn(cf_pred_matrix)
    normalized_semantic_pred_matrix = normalize_fn(semantic_pred_matrix)

    results = []
    for gamma in gamma_list:
        pred_matrix = (1 - gamma) * normalized_cf_pred_matrix + gamma * normalized_semantic_pred_matrix
        results.append(
            metric.eval(
                train_matrix=train_rate_matrix,
                pred_matrix=pred_matrix,
                test_matrix=eval_rate_matrix,
            )
        )

    best_index, best_result = find_best_result_from_results_list(results, metric_name=primary_metric)
    return gamma_list[best_index], best_result, results


def study_fixed_filters(
    filter_candidate_dict: dict[str, list[dict]],
    semantic_embeddings: torch.Tensor,
    num_users: int,
    cf_pred_matrix: torch.Tensor,
    metric: Metric,
    train_rate_matrix: torch.Tensor,
    valid_rate_matrix: torch.Tensor,
    test_rate_matrix: torch.Tensor,
    normalization_method: str = 'zscore',
    primary_metric: str = 'NDCG@20',
    plot_metric: str = 'Recall@20',
    gamma_list: list[float] | None = None,
):
    if gamma_list is None:
        gamma_list = [i * 0.01 for i in range(101)]

    normalized_cf_pred_matrix = get_normalized_cf_pred_matrix(
        cf_pred_matrix=cf_pred_matrix,
        normalization_method=normalization_method,
    )

    U, S, _ = torch.linalg.svd(semantic_embeddings, full_matrices=False)
    U_user = U[:num_users, :]
    U_item = U[num_users:, :]

    summary = {}
    fig, ax = plt.subplots(figsize=(10, 5))

    for filter_name, filter_candidates in filter_candidate_dict.items():
        candidate_valid_results = []
        candidate_test_results = []
        candidate_gamma_histories = []

        for candidate in tqdm.tqdm(filter_candidates, desc=filter_name):
            filtered_S = candidate['filter_func'](S)
            semantic_pred_matrix = build_semantic_matrix_from_filtered_s(
                U_user=U_user,
                U_item=U_item,
                filtered_S=filtered_S,
            )

            best_gamma, best_valid_result, gamma_results = search_best_gamma(
                cf_pred_matrix=cf_pred_matrix,
                semantic_pred_matrix=semantic_pred_matrix,
                train_rate_matrix=train_rate_matrix,
                eval_rate_matrix=valid_rate_matrix,
                metric=metric,
                normalization_method=normalization_method,
                primary_metric=primary_metric,
                gamma_list=gamma_list,
                normalized_cf_pred_matrix=normalized_cf_pred_matrix,
            )
            best_test_result = evaluate_cf_semantic_with_gamma(
                cf_pred_matrix=cf_pred_matrix,
                semantic_pred_matrix=semantic_pred_matrix,
                gamma=best_gamma,
                train_rate_matrix=train_rate_matrix,
                eval_rate_matrix=test_rate_matrix,
                metric=metric,
                normalization_method=normalization_method,
                normalized_cf_pred_matrix=normalized_cf_pred_matrix,
            )

            candidate_valid_results.append({
                'param_name': candidate['param_name'],
                'param_value': candidate['param_value'],
                'best_gamma': best_gamma,
                'metrics': best_valid_result,
            })
            candidate_test_results.append(best_test_result)
            candidate_gamma_histories.append(gamma_results)

        best_candidate_index, _ = find_best_result_from_results_list(
            [item['metrics'] for item in candidate_valid_results],
            metric_name=primary_metric,
        )
        best_candidate = candidate_valid_results[best_candidate_index]
        best_test_result = candidate_test_results[best_candidate_index]

        per_gamma_best = []
        for gamma_index, gamma in enumerate(gamma_list):
            results_at_gamma = [history[gamma_index] for history in candidate_gamma_histories]
            best_index_at_gamma, best_result_at_gamma = find_best_result_from_results_list(
                results_at_gamma,
                metric_name=primary_metric,
            )
            per_gamma_best.append({
                'gamma': gamma,
                'best_param_name': filter_candidates[best_index_at_gamma]['param_name'],
                'best_param_value': filter_candidates[best_index_at_gamma]['param_value'],
                'best_result': best_result_at_gamma,
            })

        summary[filter_name] = {
            'best_param_name': best_candidate['param_name'],
            'best_param_value': best_candidate['param_value'],
            'best_gamma': best_candidate['best_gamma'],
            'best_valid_result': best_candidate['metrics'],
            'best_test_result': best_test_result,
            'per_gamma_best': per_gamma_best,
        }

        ax.plot(
            range(len(filter_candidates)),
            [item['metrics'][plot_metric] for item in candidate_valid_results],
            label=filter_name,
        )

        print(
            f"{filter_name}: best {best_candidate['param_name']}={best_candidate['param_value']}, "
            f"gamma={best_candidate['best_gamma']}, valid={best_candidate['metrics']}, test={best_test_result}"
        )

    ax.set_xlabel('filter candidate index')
    ax.set_ylabel(plot_metric)
    ax.set_title(f'Fixed filter search with gamma sweep ({plot_metric})')
    ax.legend()

    return summary

In [ ]:
filter_candidate_dict = {
    'pow_high': [
        {'param_name': 'power', 'param_value': round(p, 2), 'filter_func': lambda S, p=p: torch.pow(S, p)}
        for p in [i * 0.01 for i in range(0, 101)]
    ],
    'pow_low': [
        {'param_name': 'power', 'param_value': round(p, 2), 'filter_func': lambda S, p=p: torch.pow(S, p)}
        for p in [1.0 + i * 0.02 for i in range(0, 101)]
    ],
    'log': [
        {'param_name': 'alpha', 'param_value': round(a, 2), 'filter_func': lambda S, a=a: torch.log1p(a * (S / S.max().clamp_min(1e-12)))}
        for a in [i * 0.4 for i in range(0, 101)]
    ],
    'exp': [
        {'param_name': 'alpha', 'param_value': round(a, 2), 'filter_func': lambda S, a=a: torch.exp(a * (S / S.max().clamp_min(1e-12)))}
        for a in [i * 0.08 for i in range(0, 101)]
    ],
    'rational': [
        {'param_name': 'beta', 'param_value': round(b, 2), 'filter_func': lambda S, b=b: (S / S.max().clamp_min(1e-12)) / ((S / S.max().clamp_min(1e-12)) + b)}
        for b in [0.01 + i * 0.01 for i in range(0, 101)]
    ],
    'soft_threshold': [
        {'param_name': 'threshold', 'param_value': round(t, 2), 'filter_func': lambda S, t=t: torch.clamp(S / S.max().clamp_min(1e-12) - t, min=0.0)}
        for t in [i * 0.01 for i in range(0, 101)]
    ],
}

fixed_filter_results = study_fixed_filters(
    filter_candidate_dict=filter_candidate_dict,
    semantic_embeddings=semantic_embeddings,
    num_users=datahandler.num_users,
    cf_pred_matrix=cf_pred_matrix,
    metric=metric,
    train_rate_matrix=datahandler.rate_matrix,
    valid_rate_matrix=datahandler.valid_rate_matrix,
    test_rate_matrix=datahandler.test_rate_matrix,
    normalization_method=normalization_method,
    primary_metric=primary_metric,
    plot_metric=plot_metric,
    gamma_list=gamma_list,
)

fixed_filter_results

In [ ]:
class SFilterMLP(TrainableModelBase):
    @dataclass
    class ModelConfig(TrainableModelBase.ModelConfig):
        rank: int | None = None
        hidden_dim: int = 32
        num_layers: int = 3
        dropout: float = 0.0
        similarity: Literal['dot', 'cos'] = 'dot'

    def __init__(
        self,
        semantic_embeddings: torch.Tensor,
        num_users: int,
        rate_matrix: torch.Tensor,
        model_config: ModelConfig,
        loss_config: TrainableModelBase.LossConfig,
    ):
        super().__init__(model_config=model_config, loss_config=loss_config)

        self.device = self.model_config.device
        self.rate_matrix = rate_matrix.to(self.device)
        self.num_users = num_users
        self.num_items = rate_matrix.shape[1]

        semantic_embeddings = semantic_embeddings.to(self.device)
        U, S, _ = torch.linalg.svd(semantic_embeddings, full_matrices=False)

        max_rank = S.shape[0]
        self.rank = max_rank if self.model_config.rank is None else min(self.model_config.rank, max_rank)

        self.register_buffer('user_basis', U[:self.num_users, :self.rank].contiguous())
        self.register_buffer('item_basis', U[self.num_users:, :self.rank].contiguous())
        self.register_buffer('base_singular_values', S[:self.rank].contiguous())

        layers = []
        in_dim = 1
        for _ in range(max(self.model_config.num_layers - 1, 0)):
            layers.append(nn.Linear(in_dim, self.model_config.hidden_dim))
            layers.append(nn.GELU())
            if self.model_config.dropout > 0:
                layers.append(nn.Dropout(self.model_config.dropout))
            in_dim = self.model_config.hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        self.filter_mlp = nn.Sequential(*layers)

        self._reset_parameters()

    def _reset_parameters(self):
        for module in self.filter_mlp:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

        last_linear = None
        for module in reversed(self.filter_mlp):
            if isinstance(module, nn.Linear):
                last_linear = module
                break

        if last_linear is not None:
            identity_bias = torch.log(torch.expm1(torch.tensor(1.0))).item()
            nn.init.zeros_(last_linear.weight)
            nn.init.constant_(last_linear.bias, identity_bias)

    def get_filtered_s(self) -> torch.Tensor:
        s_input = self.base_singular_values / self.base_singular_values.max().clamp_min(1e-12)
        gain = F.softplus(self.filter_mlp(s_input.unsqueeze(-1))).squeeze(-1)
        return self.base_singular_values * gain

    def _get_all_embeddings(self) -> tuple[torch.Tensor, torch.Tensor]:
        filtered_s = self.get_filtered_s()
        user_embedding = self.user_basis * filtered_s
        item_embedding = self.item_basis * filtered_s
        return user_embedding, item_embedding

    def forward(
        self,
        user_index: torch.Tensor,
        item_index: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        all_user_embedding, all_item_embedding = self._get_all_embeddings()
        return all_user_embedding[user_index], all_item_embedding[item_index]

    @torch.no_grad()
    def predict(self) -> torch.Tensor:
        all_user_embedding, all_item_embedding = self._get_all_embeddings()
        match self.model_config.similarity:
            case 'cos':
                all_user_embedding = F.normalize(all_user_embedding, p=2, dim=1)
                all_item_embedding = F.normalize(all_item_embedding, p=2, dim=1)
            case 'dot':
                pass
            case _:
                raise ValueError(f'Unsupported similarity: {self.model_config.similarity}')
        return all_user_embedding @ all_item_embedding.T


def train_semantic_with_gamma_search(
    datahandler: DataHandler,
    metric: Metric,
    model: TrainableModelBase,
    cf_pred_matrix: torch.Tensor,
    num_epochs: int = 120,
    num_steps: int = 20,
    patience_steps: int = 20,
    primary_metric: str = 'NDCG@20',
    normalization_method: str = 'zscore',
    gamma_list: list[float] | None = None,
    verbose: bool = True,
    use_amp: bool = True,
):
    if gamma_list is None:
        gamma_list = [i * 0.01 for i in range(101)]

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3,
        weight_decay=1e-4,
    )
    device = model.model_config.device
    best_state_dict = deepcopy(model.state_dict())
    now_patience = 0

    normalized_cf_pred_matrix = get_normalized_cf_pred_matrix(
        cf_pred_matrix=cf_pred_matrix,
        normalization_method=normalization_method,
    )

    model.eval()
    print('Initial Combined Performance (Before Training):')
    init_gamma, init_validation_result, _ = search_best_gamma(
        cf_pred_matrix=cf_pred_matrix,
        semantic_pred_matrix=model.predict(),
        train_rate_matrix=datahandler.rate_matrix,
        eval_rate_matrix=datahandler.valid_rate_matrix,
        metric=metric,
        normalization_method=normalization_method,
        primary_metric=primary_metric,
        gamma_list=gamma_list,
        normalized_cf_pred_matrix=normalized_cf_pred_matrix,
    )
    best_validation_result = init_validation_result
    best_gamma = init_gamma
    print('  ' + ', '.join([f"{k}: {v:.4f}" for k, v in init_validation_result.items()]) + f"  gamma={init_gamma}")

    report_interval = max(1, num_epochs // num_steps)
    epoch_start_time = time.time()
    accumulated_debug_info = TrainingDebugInfo()

    for epoch in range(num_epochs):
        avg_loss, debug_info = train_epoch(
            datahandler=datahandler,
            optimizer=optimizer,
            model=model,
            device=device,
            use_amp=use_amp,
            verbose=verbose,
        )

        accumulated_debug_info.data_wait_time += debug_info.data_wait_time
        accumulated_debug_info.zero_grad_time += debug_info.zero_grad_time
        accumulated_debug_info.get_loss_time += debug_info.get_loss_time
        accumulated_debug_info.backward_time += debug_info.backward_time
        accumulated_debug_info.step_time += debug_info.step_time
        accumulated_debug_info.other_time += debug_info.other_time

        if (epoch + 1) % report_interval == 0 or epoch == num_epochs - 1:
            model.eval()
            eval_start = time.time()
            step_gamma, validation_result, _ = search_best_gamma(
                cf_pred_matrix=cf_pred_matrix,
                semantic_pred_matrix=model.predict(),
                train_rate_matrix=datahandler.rate_matrix,
                eval_rate_matrix=datahandler.valid_rate_matrix,
                metric=metric,
                normalization_method=normalization_method,
                primary_metric=primary_metric,
                gamma_list=gamma_list,
                normalized_cf_pred_matrix=normalized_cf_pred_matrix,
            )
            accumulated_debug_info.eval_time += time.time() - eval_start
            epoch_elapsed = time.time() - epoch_start_time

            metrics_str = ', '.join([f"{k}: {v:.4f}" for k, v in validation_result.items()])
            known = (
                accumulated_debug_info.data_wait_time
                + accumulated_debug_info.zero_grad_time
                + accumulated_debug_info.get_loss_time
                + accumulated_debug_info.backward_time
                + accumulated_debug_info.step_time
                + accumulated_debug_info.eval_time
            )
            residual_other = max(epoch_elapsed - known, 0.0)
            time_breakdown = (
                f"Data:{accumulated_debug_info.data_wait_time:.1f}s "
                f"ZG:{accumulated_debug_info.zero_grad_time:.1f}s "
                f"Loss:{accumulated_debug_info.get_loss_time:.1f}s "
                f"BW:{accumulated_debug_info.backward_time:.1f}s "
                f"Step:{accumulated_debug_info.step_time:.1f}s "
                f"Eval:{accumulated_debug_info.eval_time:.1f}s "
                f"Other:{residual_other:.1f}s"
            ) if verbose else ''
            time_str = f"Time: {epoch_elapsed:.1f}s {time_breakdown}"
            print(f"Epoch [{epoch + 1:>5d}/{num_epochs}]  Loss: {avg_loss:.4f}  {time_str}  {metrics_str}  gamma={step_gamma}")

            if validation_result[primary_metric] > best_validation_result[primary_metric]:
                best_validation_result = deepcopy(validation_result)
                best_state_dict = deepcopy(model.state_dict())
                best_gamma = step_gamma
                now_patience = 0
            else:
                now_patience += 1
                if now_patience > patience_steps:
                    print(f"\n{'=' * 80}")
                    print(f"Early stopping at epoch {epoch + 1}")
                    print('Best validation results (CF + semantic):')
                    print(f"  gamma: {best_gamma}")
                    print('  ' + ', '.join([f"{k}: {v:.4f}" for k, v in best_validation_result.items()]))
                    print(f"{'=' * 80}")
                    break

            epoch_start_time = time.time()
            accumulated_debug_info = TrainingDebugInfo()

    print('\nLoading best model and evaluating on test set...')
    model.load_state_dict(best_state_dict)
    model.eval()
    test_result = evaluate_cf_semantic_with_gamma(
        cf_pred_matrix=cf_pred_matrix,
        semantic_pred_matrix=model.predict(),
        gamma=best_gamma,
        train_rate_matrix=datahandler.rate_matrix,
        eval_rate_matrix=datahandler.test_rate_matrix,
        metric=metric,
        normalization_method=normalization_method,
        normalized_cf_pred_matrix=normalized_cf_pred_matrix,
    )
    print(f"{'=' * 80}")
    print('Final test results (CF + semantic):')
    print(f"  gamma: {best_gamma}")
    print('  ' + ', '.join([f"{k}: {v:.4f}" for k, v in test_result.items()]))
    print(f"{'=' * 80}")

    return {
        'best_gamma': best_gamma,
        'best_validation_result': best_validation_result,
        'test_result': test_result,
    }


datahandler.batch_size = 4096 if device == 'cuda' else 512
datahandler.num_neg_item = 1
datahandler.use_fast_sampling = True
datahandler._train_dataloader = None

semantic_model = SFilterMLP(
    semantic_embeddings=semantic_embeddings,
    num_users=datahandler.num_users,
    rate_matrix=datahandler.rate_matrix,
    model_config=SFilterMLP.ModelConfig(
        device=device,
        rank=None,
        hidden_dim=32,
        num_layers=3,
        dropout=0.0,
        similarity='dot',
    ),
    loss_config=TrainableModelBase.InfoNCELossConfig(
        temperature=0.1,
        is_inbatch=True,
    ),
).to(device)

semantic_train_result = train_semantic_with_gamma_search(
    datahandler=datahandler,
    metric=metric,
    model=semantic_model,
    cf_pred_matrix=cf_pred_matrix,
    num_epochs=120,
    num_steps=20,
    patience_steps=20,
    primary_metric=primary_metric,
    normalization_method=normalization_method,
    gamma_list=gamma_list,
    verbose=True,
    use_amp=(device == 'cuda'),
)

with torch.no_grad():
    base_s = semantic_model.base_singular_values.detach().cpu()
    learned_filtered_s = semantic_model.get_filtered_s().detach().cpu()

plt.figure(figsize=(8, 4))
plt.plot(base_s.numpy(), label='base S')
plt.plot(learned_filtered_s.numpy(), label='learned filtered S')
plt.legend()
plt.title('Learned trainable semantic filter')
plt.show()

semantic_train_result
